# NEURAL RMF: reproducible EEG monitoring demonstration

This notebook runs a complete example of **individualized EEG monitoring** on two public CHB-MIT recordings. Each file is treated as an independent session: an eight-minute reference is first captured, then 30-second windows are processed.

The goal is to observe a risk trajectory and compare it, **after analysis**, with the public seizure-onset annotations. Labels are not used for calibration or state generation. This notebook does not predict an exact seizure time, diagnose epilepsy, or constitute a medical device.

**Included cases:** `chb01_03` and `chb03_04`. Both are downloaded on demand from PhysioNet; the full dataset is not downloaded.


## 1. Install the research edition

The library is distributed through binary wheels to protect the Resonant Memory Field engine. This cell selects the Linux wheel compatible with the Colab Python runtime. If Colab changes to an unsupported version, the cell stops with a clear explanation.


In [ ]:
import subprocess, sys

WHEELS = {
    "cp312": "https://github.com/Gusmal02/NEURAL-RMF/releases/download/v0.1.0/neural_rmf-0.1.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl",
    "cp313": "https://github.com/Gusmal02/NEURAL-RMF/releases/download/v0.1.0/neural_rmf-0.1.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl",
}
tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
assert tag in WHEELS, f"No published wheel is available for Python {sys.version.split()[0]}. Available: {', '.join(WHEELS)}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", WHEELS[tag]])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pandas", "matplotlib", "requests"])

import neural_rmf
print("NEURAL RMF installed:", neural_rmf.__version__)


## 2. Define the two recordings and download only what is needed

CHB-MIT provides the EDF recordings and summary files publicly. The download function validates the file size so that a partial download is never reused. Labels are read only for retrospective comparison and are drawn at the end as vertical lines.


In [ ]:
from pathlib import Path
import re
import requests

DATA = Path("chbmit_demo_data")
DATA.mkdir(exist_ok=True)

CASES = [
    {"patient": "chb01", "file": "chb01_03.edf", "profile": "gradual trajectory example"},
    {"patient": "chb03", "file": "chb03_04.edf", "profile": "abrupt trajectory example"},
]

def download_verified(url, destination):
    # Descarga a un temporal y valida Content-Length cuando existe.
    destination = Path(destination)
    response = requests.get(url, stream=True, timeout=90)
    response.raise_for_status()
    expected = int(response.headers.get("Content-Length", 0))
    tmp = destination.with_suffix(destination.suffix + ".part")
    written = 0
    with open(tmp, "wb") as handle:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                handle.write(chunk)
                written += len(chunk)
    if expected and written != expected:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f"Incomplete download for {destination.name}: {written} de {expected} bytes.")
    tmp.replace(destination)
    return destination

def annotation_block(summary_text, filename):
    start = summary_text.find(f"File Name: {filename}")
    if start < 0:
        return "The file was not found in the summary.", []
    end = summary_text.find("\n\n", start)
    block = summary_text[start:] if end < 0 else summary_text[start:end]
    onsets = [int(x) for x in re.findall(r"Seizure Start Time:\s*(\d+)\s*seconds", block)]
    return block, onsets

for case in CASES:
    folder = DATA / case["patient"]
    folder.mkdir(exist_ok=True)
    case["edf"] = folder / case["file"]
    case["summary"] = folder / f"{case['patient']}-summary.txt"
    base = f"https://physionet.org/files/chbmit/1.0.0/{case['patient']}"
    if not case["edf"].exists():
        print("Downloading", case["file"], "(about 40 MB)...")
        download_verified(f"{base}/{case['file']}", case["edf"])
    if not case["summary"].exists():
        download_verified(f"{base}/{case['patient']}-summary.txt", case["summary"])
    block, onsets = annotation_block(case["summary"].read_text(errors="ignore"), case["file"])
    case["onsets_sec"] = onsets
    print(f"\n{case['file']} — {case['profile']}")
    print(block)


## 3. Configure the session and interpretation logic

The four bipolar derivations are `F7-T7`, `T7-P7`, `F8-T8`, and `T8-P8`. The P80 threshold is calculated exclusively from the first eight minutes of each EDF and remains fixed.

The red condition is not simply high novelty: it requires sustained loading, a sufficient peak, descent from that peak, increasing field coherence, and persistence. The visible implementation below is the interpretable state layer; the binary RMF engine calculates the field metrics.


In [ ]:
import numpy as np
import pandas as pd
from neural_rmf import calibrate, detect

CHANNELS = ["F7-T7", "T7-P7", "F8-T8", "T8-P8"]
CALIB_MINUTES = 8.0
WINDOW_SECONDS = 30.0
STATE_COLOR = {"calibracion": "#BFC9CA", "verde": "#2ECC71", "amarillo": "#F1C40F", "naranja": "#F39C12", "rojo": "#E74C3C"}

class TrajectorySemaphore:
    # Interpretable state: loading -> peak -> decline + coherence -> red.
    def __init__(self, p80, peak_factor=2.0, n_orange=3, n_red=3, r_window=5, drop_min_pct=0.10, n_close=6):
        self.p80 = float(p80)
        self.peak_factor, self.n_orange, self.n_red = peak_factor, n_orange, n_red
        self.r_window, self.drop_min_pct, self.n_close = r_window, drop_min_pct, n_close
        self.state, self.above, self.below, self.confirm, self.peak, self.r_history = "verde", 0, 0, 0, 0.0, []

    def update(self, novelty_max, r_field):
        novelty_max, r_field = float(novelty_max), float(r_field)
        self.r_history.append(r_field)
        self.r_history = self.r_history[-self.r_window:]
        r_slope = self.r_history[-1] - self.r_history[0] if len(self.r_history) == self.r_window else 0.0
        if self.state in ("verde", "amarillo"):
            self.above = self.above + 1 if novelty_max > self.p80 else 0
            if self.above >= self.n_orange:
                self.state, self.peak, self.below, self.confirm = "naranja", novelty_max, 0, 0
            elif self.above:
                self.state = "amarillo"
            else:
                self.state = "verde"
        else:
            self.peak = max(self.peak, novelty_max)
            signature = (self.peak >= self.peak_factor * self.p80 and
                         novelty_max <= self.peak * (1.0 - self.drop_min_pct) and
                         r_slope > 0)
            self.confirm = self.confirm + 1 if signature else 0
            self.state = "rojo" if self.confirm >= self.n_red else "naranja"
            self.below = self.below + 1 if novelty_max <= self.p80 else 0
            if self.below >= self.n_close and not signature:
                self.state, self.above, self.below, self.confirm, self.peak = "verde", 0, 0, 0, 0.0
        return self.state, r_slope


## 4. Run each case independently

The label does not enter the field or the state machine. It is used only afterward to measure the temporal distance between a red alert and the annotated onset. If a red trajectory does not coincide with a label, the notebook retains it as a review candidate; it is not automatically classified as either a seizure or a false alarm.


In [ ]:
def run_case(case):
    model = calibrate(str(case["edf"]), calib_minutes=CALIB_MINUTES, channels=CHANNELS)
    # detect() produce las métricas de todas las ventanas. El semáforo demostrativo
    # comienza solo tras la calibración, por lo que la referencia no se cuenta como alerta.
    result = detect(model, str(case["edf"]))
    n = min(len(result.novelty_por_minuto), len(result.nov_collective), len(result.r_field_series))
    frame = pd.DataFrame({
        "window": np.arange(n),
        "t_start_sec": np.arange(n) * WINDOW_SECONDS,
        "t_end_sec": (np.arange(n) + 1) * WINDOW_SECONDS,
        "novelty_max": np.asarray(result.novelty_por_minuto[:n], dtype=float),
        "novelty_col": np.asarray(result.nov_collective[:n], dtype=float),
        "r_field": np.asarray(result.r_field_series[:n], dtype=float),
    })
    frame["slope_r"] = frame["r_field"].rolling(5, min_periods=5).apply(lambda x: x.iloc[-1] - x.iloc[0], raw=False).fillna(0.0)
    frame["state"] = "calibracion"
    calibration_windows = int(CALIB_MINUTES * 60 / WINDOW_SECONDS)
    sem = TrajectorySemaphore(result.umbral_calibrado)
    for idx in range(calibration_windows, len(frame)):
        state, _ = sem.update(frame.at[idx, "novelty_max"], frame.at[idx, "r_field"])
        frame.at[idx, "state"] = state

    red = frame[frame["state"].eq("rojo")]
    first_red_sec = float(red.iloc[0]["t_start_sec"]) if len(red) else None
    first_onset = case["onsets_sec"][0] if case["onsets_sec"] else None
    lead_min = (first_onset - first_red_sec) / 60 if first_red_sec is not None and first_onset is not None else None
    summary = {
        "case": case["file"], "profile": case["profile"], "channels": CHANNELS,
        "calibration_minutes": CALIB_MINUTES, "p80_baseline_threshold": float(result.umbral_calibrado),
        "labeled_onsets_sec": case["onsets_sec"], "first_red_sec": first_red_sec,
        "lead_time_min_first_labeled_onset": lead_min,
        "red_minutes_after_calibration": float((frame["state"] == "rojo").sum() * WINDOW_SECONDS / 60),
        "state_minutes_after_calibration": {state: float((frame.loc[frame["state"] != "calibracion", "state"] == state).sum() * WINDOW_SECONDS / 60) for state in ["verde", "amarillo", "naranja", "rojo"]},
    }
    return frame, summary

RUNS = {}
SUMMARIES = []
for case in CASES:
    print("Processing", case["file"], "...")
    frame, summary = run_case(case)
    RUNS[case["file"]] = frame
    SUMMARIES.append(summary)
    lead_text = "no preceding red state" if summary["lead_time_min_first_labeled_onset"] is None else f"{summary['lead_time_min_first_labeled_onset']:.1f} min"
    print("  P80:", f"{summary['p80_baseline_threshold']:.4f}", "| first red:", summary["first_red_sec"], "s | lead time:", lead_text)


## 5. Visualize the trajectories and compare them with annotations

The gray band is calibration. Red vertical lines are onset annotations from the public dataset. The bottom strip represents the traffic-light output. The temporal relationship is displayed for inspection; it does not make the label an algorithm input.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def plot_case(case, frame, summary):
    t = frame["t_start_sec"] / 60
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.6, .8]})
    fig.suptitle(f"NEURAL RMF — {case['file']} ({case['profile']})", fontweight="bold")
    axes[0].plot(t, frame["novelty_max"], label="Novelty Max", color="#2874A6", lw=1.25)
    axes[0].plot(t, frame["novelty_col"], label="Novelty Col", color="#7D3C98", lw=1.05, alpha=.85)
    axes[0].axhline(summary["p80_baseline_threshold"], color="#B9770E", ls=":", label="Baseline P80 threshold")
    axes[0].set_ylabel("Change relative to baseline")
    axes[0].legend(ncol=3, fontsize=9)
    axes[1].plot(t, frame["r_field"], label="r_field (coherencia del campo)", color="#148F77", lw=1.2)
    axes[1].plot(t, frame["slope_r"], label="r_field change", color="#CA6F1E", lw=.9, alpha=.8)
    axes[1].axhline(0, color="black", lw=.6)
    axes[1].set_ylabel("Coherence / change")
    axes[1].legend(fontsize=9)
    for _, row in frame.iterrows():
        axes[2].barh(0, WINDOW_SECONDS / 60, left=row["t_start_sec"] / 60, height=.8, color=STATE_COLOR[row["state"]])
    axes[2].set_yticks([])
    axes[2].set_ylabel("State")
    axes[2].set_xlabel("Time from EDF start (minutes)")
    axes[2].legend(handles=[Patch(color=c, label=s.title()) for s, c in STATE_COLOR.items()], ncol=5, fontsize=8, loc="upper left")
    for ax in axes:
        ax.axvspan(0, CALIB_MINUTES, color="#D5D8DC", alpha=.35, label="Calibration" if ax is axes[0] else None)
        for onset in case["onsets_sec"]:
            ax.axvline(onset / 60, color="#C0392B", ls="--", lw=1.4)
        ax.grid(alpha=.22)
    fig.text(.5, .01, "Vertical red lines: CHB-MIT annotated onset. This figure is a retrospective research comparison.", ha="center", fontsize=8)
    plt.tight_layout(rect=(0, .03, 1, .96))
    return fig

OUTPUT = Path("neural_rmf_colab_outputs")
OUTPUT.mkdir(exist_ok=True)
for case in CASES:
    frame = RUNS[case["file"]]
    summary = next(x for x in SUMMARIES if x["case"] == case["file"])
    fig = plot_case(case, frame, summary)
    fig.savefig(OUTPUT / f"{case['file'].replace('.edf', '')}_timeline.png", dpi=170, bbox_inches="tight")
    plt.show()


## 6. Read the comparative summary

A positive lead time means the first red state occurred before the first annotated onset. An unlabeled trajectory remains a research candidate: its interpretation requires signal quality assessment, clinical context, video-EEG, and expert review.


In [ ]:
summary_table = pd.DataFrame(SUMMARIES)[[
    "case", "profile", "p80_baseline_threshold", "labeled_onsets_sec",
    "first_red_sec", "lead_time_min_first_labeled_onset", "red_minutes_after_calibration"
]]
display(summary_table)

for item in SUMMARIES:
    print(f"\n{item['case']}")
    if item["lead_time_min_first_labeled_onset"] is not None:
        print(f"  The first red alert occurred {item['lead_time_min_first_labeled_onset']:.1f} min before the first annotated onset.")
    else:
        print("  No preceding red state was available for the first annotated onset under this configuration.")
    print("  States posteriores a calibración (min):", item["state_minutes_after_calibration"])


## 7. Export the reproducible report

The notebook saves one CSV timeline per case, a JSON file with parameters and results, PNG figures, and a downloadable ZIP archive. These outputs do not replace clinical EEG interpretation or prospective validation.


In [ ]:
import json, shutil
from datetime import datetime, timezone

for case in CASES:
    RUNS[case["file"]].to_csv(OUTPUT / f"{case['file'].replace('.edf', '')}_timeline.csv", index=False)

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Reproducible technical demonstration of individualized EEG monitoring; not diagnosis or deterministic prediction.",
    "dataset": "CHB-MIT Scalp EEG Database, PhysioNet",
    "parameters": {"channels": CHANNELS, "calibration_minutes": CALIB_MINUTES, "window_seconds": WINDOW_SECONDS,
                   "red_signature": {"peak_factor": 2.0, "minimum_orange_windows": 3, "minimum_red_windows": 3, "novelty_drop_min_pct": 0.10, "r_field_history_windows": 5}},
    "cases": SUMMARIES,
}
(OUTPUT / "neural_rmf_colab_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
zip_path = shutil.make_archive("neural_rmf_colab_outputs", "zip", OUTPUT)
print("Generated files:")
for path in sorted(OUTPUT.iterdir()):
    print(" -", path.name)

from google.colab import files
files.download(zip_path)


## Responsible interpretation

This example shows how to reproduce session-specific monitoring and compare its trajectories with public annotations. Red indicates a high-risk trajectory under this demonstration's parameters; it does not confirm that a seizure is occurring or guarantee that one will occur. Unlabeled trajectories may reflect physiological changes, artifacts, interrupted transitions, or subtle epileptic activity. Resolving those possibilities requires prospective clinical evaluation with additional context.

To reproduce the workflow with another EDF, change only the `CASES` list; keep a new calibration for every session and check that the four derivations are available with the same labels.
